# Data Transformation Visualization: Intermediate Activations

Captures real intermediate activations from a trained `exp2b_flash_learned_pool` checkpoint  
to produce the 6-stage data transformation strip visualization.

**Stages:**
1. Raw clinical codes (text, one day)
2. After embedding lookup → `[80, 256]`
3. After Learned Attention Pooling → `[1, 256]`
4. All days stacked (after demo injection) → `[200, 256]`
5. After 6 temporal layers → `[200, 256]`
6. Final member embedding → `[256]`

Run on GCP Vertex AI where the trained checkpoint is available.

In [ ]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

# Add core module to path (adjust if running from a different working directory)
sys.path.insert(0, '/path/to/Clinical_TE/dev/moe')
from moe_flashattn_4_core import FlashAttentionConfig, FlashAttentionTransformer

## 1. Load Model and Sample Data

In [ ]:
# ---- Configuration ----
checkpoint_path = 'path/to/exp2b_round10_best.pt'  # UPDATE THIS
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---- Load model ----
# Replace with your actual model loading code from the training notebook.
# Example skeleton:
#
# config = FlashAttentionConfig(
#     len_dy=200, len_cd=80, cd_cnt=75516, target_cd_cnt=6297,
#     embedding_size=256, nlayers=6
# )
# model = FlashAttentionTransformer(config).to(device)
# checkpoint = torch.load(checkpoint_path, map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()
# print('Model loaded.')

# ---- Load one sample ----
# Replace with your actual dataset loading code.
# Example skeleton:
#
# dataset = ...  # your ClinicalDataset or ClinicalDatasetLazy
# sample = dataset[0]          # or pick a member with rich history
# cd_tensor = sample['cd'].unsqueeze(0).to(device)   # [1, 200, 80]
# dt_cnt = int(sample['dt_cnt'])                     # number of valid days
# demo = sample.get('demo', None)                    # demographics if used
# if demo is not None:
#     demo = demo.unsqueeze(0).to(device)
# print(f'Sample loaded: {dt_cnt} valid days')

## 2. Register Forward Hooks

In [ ]:
activations = {}

def make_hook(name):
    def hook(module, input, output):
        if isinstance(output, tuple):
            activations[name] = output[0].detach().cpu()
        else:
            activations[name] = output.detach().cpu()
    return hook

hooks = []

# Stage 2: After embedding lookup — output shape [batch, len_dy, len_cd, embedding_size]
hooks.append(model.code_embedding.register_forward_hook(make_hook('stage2_embeddings')))

# Stage 3: After Learned Attention Pooling — output shape [batch, len_dy, embedding_size]
if hasattr(model, 'daily_pooling'):
    hooks.append(model.daily_pooling.register_forward_hook(make_hook('stage3_after_lap')))

# Stage 4: After demographic injection / before temporal encoder
# Uncomment and adjust attribute name to match your model:
# hooks.append(model.fuse_embedding.register_forward_hook(make_hook('stage4_after_demo')))

# Stage 5: After each of the 6 temporal layers
for i, layer in enumerate(model.temporal_layers):
    hooks.append(layer.register_forward_hook(make_hook(f'stage5_layer_{i}')))

print(f'Registered {len(hooks)} hooks.')

## 3. Forward Pass

In [ ]:
with torch.no_grad():
    # Replace with your actual forward call signature.
    # Examples:
    # output = model(cd_tensor)
    # output = model(cd_tensor, demo=demo)
    pass

# Remove hooks immediately after the forward pass
for h in hooks:
    h.remove()

print('Captured activation keys:', list(activations.keys()))

## 4. Inspect Raw Activation Shapes

In [ ]:
for key, tensor in activations.items():
    print(f'{key:35s}  shape={tuple(tensor.shape)}')

## 5. Main Visualization: 6-Stage Transformation Strip

In [ ]:
def plot_heatmap(ax, data, title, ylabel='', xlabel='256 dims', cmap='RdBu_r'):
    """Plot a 2D tensor as a diverging heatmap centered at zero."""
    if data.dim() == 3:
        data = data[0]  # drop batch dim
    d = data.numpy()
    vmax = max(abs(d.min()), abs(d.max()))
    vmin = -vmax
    vcenter = 0.0
    if vmin == vcenter or vcenter == vmax:
        norm = None
    else:
        norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
    ax.imshow(d, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xlabel(xlabel, fontsize=7)
    ax.tick_params(labelsize=6)


# Choose a day index with real codes (not all-padding)
day_idx = 0  # adjust if day 0 has no codes

fig = plt.figure(figsize=(26, 10))
gs = gridspec.GridSpec(2, 6, height_ratios=[1, 4], hspace=0.35, wspace=0.35)

# --- Stage 2: Embedding lookup [80, 256] (one day) ---
ax1 = fig.add_subplot(gs[:, 0])
emb_data = activations['stage2_embeddings'][0, day_idx]  # [80, 256]
plot_heatmap(ax1, emb_data, f'Stage 2: Embedding Lookup\n[80, 256]  Day {day_idx}',
             ylabel='80 code slots')

# --- Stage 3: After LAP [1, 256] (same day) ---
ax2 = fig.add_subplot(gs[0, 1])
lap_data = activations['stage3_after_lap'][0, day_idx:day_idx+1]  # [1, 256]
plot_heatmap(ax2, lap_data, 'Stage 3: After LAP\n[1, 256]', ylabel='1 day')

# --- Stage 4: All days after demo injection [200, 256] ---
ax3 = fig.add_subplot(gs[:, 2])
demo_data = activations.get('stage4_after_demo', activations['stage3_after_lap'])
plot_heatmap(ax3, demo_data[0], 'Stage 4: All Days (pre-temporal)\n[200, 256]',
             ylabel='200 days')

# --- Stage 5a: After temporal layer 0 [200, 256] ---
ax4 = fig.add_subplot(gs[:, 3])
plot_heatmap(ax4, activations['stage5_layer_0'][0],
             'Stage 5a: After Temporal Layer 0\n[200, 256]', ylabel='200 days')

# --- Stage 5b: After temporal layer 5 (final) [200, 256] ---
ax5 = fig.add_subplot(gs[:, 4])
plot_heatmap(ax5, activations['stage5_layer_5'][0],
             'Stage 5b: After Temporal Layer 5\n[200, 256]', ylabel='200 days')

# --- Stage 6: Final member embedding [1, 256] ---
ax6 = fig.add_subplot(gs[0, 5])
last_day = dt_cnt - 1
final_emb = activations['stage5_layer_5'][0, last_day:last_day+1]  # [1, 256]
plot_heatmap(ax6, final_emb, f'Stage 6: Member Embedding\n[1, 256]  Day {last_day}', ylabel='')

# Stage labels along the bottom row (second subplot row, col 1 and 5)
for ax in [ax2, ax6]:
    ax.set_aspect('auto')

plt.suptitle('Data Transformation Through Clinical TE Architecture\n(exp2b_flash_learned_pool)',
             fontsize=13, fontweight='bold', y=1.01)

out_path = 'data_transformation_strip.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

## 6. Stage 4 vs Stage 5 Side-by-Side (Key Contrast Slide)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

demo_data_np = activations.get('stage4_after_demo', activations['stage3_after_lap'])[0].numpy()
final_np = activations['stage5_layer_5'][0].numpy()

vmax = max(abs(demo_data_np).max(), abs(final_np).max())
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

axes[0].imshow(demo_data_np, aspect='auto', cmap='RdBu_r', norm=norm, interpolation='nearest')
axes[0].set_title('Stage 4: Pre-Temporal Encoder\n[200, 256] — rows look independent',
                  fontsize=11, fontweight='bold')
axes[0].set_ylabel('200 days (rows)', fontsize=10)
axes[0].set_xlabel('256 embedding dims', fontsize=10)

im = axes[1].imshow(final_np, aspect='auto', cmap='RdBu_r', norm=norm, interpolation='nearest')
axes[1].set_title('Stage 5: After 6 Temporal Layers\n[200, 256] — vertical stripes, temporal continuity',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('256 embedding dims', fontsize=10)

fig.colorbar(im, ax=axes, shrink=0.6, label='Activation value')

plt.suptitle('The Key Contrast: What 6 Causal Attention Layers Learn',
             fontsize=13, fontweight='bold', y=1.02)

out_path2 = 'stage4_vs_stage5_contrast.png'
plt.savefig(out_path2, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path2}')

## 7. Attention Weights for Stage 3 (LAP)

If `daily_pooling` exposes attention weights, visualize which codes the model attends to on a given day.

In [ ]:
# Only works if the LAP module stores attention weights as an attribute after forward.
# Adjust attribute name to match LearnedAttentionPooling implementation.

if hasattr(model, 'daily_pooling') and hasattr(model.daily_pooling, 'last_attn_weights'):
    attn = model.daily_pooling.last_attn_weights  # expected [batch, len_dy, len_cd] or [batch, len_dy, 1, len_cd]
    if attn.dim() == 4:
        attn = attn.squeeze(2)  # [batch, len_dy, len_cd]
    attn_day = attn[0, day_idx].cpu().numpy()  # [80]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(range(len(attn_day)), attn_day[::-1], color='steelblue')
    ax.set_xlabel('Attention weight', fontsize=10)
    ax.set_ylabel('Code slot (80 → 1)', fontsize=10)
    ax.set_title(f'LAP Attention Weights — Day {day_idx}\n(higher = model weighted this code more)',
                 fontsize=11)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('lap_attention_weights.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved: lap_attention_weights.png')
else:
    print('Attention weights not exposed by daily_pooling. '
          'Add a self.last_attn_weights attribute in LearnedAttentionPooling.forward() to enable this cell.')

## 8. Layer-by-Layer Evolution (Animated or Subplots)

Shows how the representation changes after each of the 6 temporal layers.

In [ ]:
n_layers = sum(1 for k in activations if k.startswith('stage5_layer_'))
print(f'Found {n_layers} temporal layer activations.')

fig, axes = plt.subplots(1, n_layers, figsize=(4 * n_layers, 8), sharey=True)
if n_layers == 1:
    axes = [axes]

all_data = [activations[f'stage5_layer_{i}'][0].numpy() for i in range(n_layers)]
vmax = max(abs(d).max() for d in all_data)
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

for i, (ax, d) in enumerate(zip(axes, all_data)):
    ax.imshow(d, aspect='auto', cmap='RdBu_r', norm=norm, interpolation='nearest')
    ax.set_title(f'Layer {i}', fontsize=10, fontweight='bold')
    ax.set_xlabel('256 dims', fontsize=8)
    if i == 0:
        ax.set_ylabel('200 days', fontsize=9)
    ax.tick_params(labelsize=6)

plt.suptitle('Representation After Each Temporal Layer\n(left=shallow, right=deep)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('temporal_layer_evolution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: temporal_layer_evolution.png')